## Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

In [3]:
from funcoes_escoragem import *

## Diretório

In [4]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Funil Modelos

In [5]:
project_id = 'loft-dl-fintech'

In [6]:
query = '''
WITH tb_leads AS (
  SELECT 
    rf.contract_id,
    date(rf.dt_lead) as dt_lead,
    date(cf.requested_at) as requested_at,
    date(rf.dt_proposta_iniciada) as iniciada_at,
    date(rf.dt_proposta_enviada) as enviada_at,
    date(rf.activated_at) as activated_at,
    date(rf.cancelled_at) as cancelled_at,
    date(rf.dt_saida) as dt_saida,
    cf.tipo_contrato,
    rd.product_nm,
    cf.modeloBlend,
    cf.bureau_nm,
    cf.modelo_blend,
    case when cf.modeloBlend in ('BLEND_4', 'BVS_CUSTOM_V2', 'HFT1') then 'BLEND4'
         when cf.bureau_nm in ('BLEND_4', 'BVS_CUSTOM_V2', 'HFT1') then 'BLEND4'
         when cf.modeloBlend in ('BLEND3_3', 'BVS_CUSTOM', 'HVA4') then 'BLEND3'
         when cf.bureau_nm in ('BLEND3_3', 'BVS_CUSTOM', 'HVA4') then 'BLEND3'
         else modeloBlend end as bureau_nm_ajust,
    case when cf.modeloBlend in ('BVS_CUSTOM_V2', 'HFT1', 'BVS_CUSTOM', 'HVA4') then 1
         when cf.bureau_nm in ('BVS_CUSTOM_V2', 'HFT1', 'BVS_CUSTOM', 'HVA4') then 1 else 0 end as is_fallback,
    cast(rf.total_rental_value_informed_nr as FLOAT64) as total_rental_value_informed_nr,
    cast(rf.rental_value_nr as FLOAT64) as rental_value_nr,
    cf.agency_id,
    cf.contract_city_nm,
    cf.person_age,
    cf.qtd_proponentes,
    cf.score_imobiliaria,
    cf.person_restriction_total_value,
    ca.bvs_cust_score_nr,
    ca.blend_regressao_predict_nr,
    cf.rating_score_ds,
    cf.person_deemed_income,
    rd.pre_analysis_result,
    rd.lead_elegivel,
    rd.proposta_iniciada,
    rd.proposta_enviada,
    rd.proposta_aprovada,
    rd.proposta_ativada,
    rd.is_activeted
  FROM loft-dl-fintech.cp_gold.requests_fact AS rf
  LEFT JOIN loft-dl-fintech.cp_gold.requests_dim AS rd
    ON rf.contract_id = rd.contract_id
  LEFT JOIN loft-dl-fintech.cp_gold.credit_fact AS cf
    ON rf.contract_id = cf.contract_id
  LEFT JOIN loft-dl-fintech.cp_silver.int_credit_analyses AS ca
    ON rf.contract_id = ca.contract_id
  WHERE cf.tipo_contrato = 'PF'
),
first_defaults AS (
  SELECT
    contract_id
    , DATE(MIN(pendency_created_at)) AS first_comunicacao_date
    , DATE(MIN(pendency_at)) AS first_competencia_date
  FROM loft-dl-fintech.cp_gold.watchlist_fact
  WHERE pendency_type = "Inadimplência"
  GROUP BY contract_id
)
SELECT
  l.*
  , fd.first_comunicacao_date
  , fd.first_competencia_date
FROM tb_leads AS l
LEFT JOIN first_defaults as fd
  on l.contract_id = fd.contract_id
  and l.activated_at is not null
where 1=1
    and date(l.requested_at) >= date('2026-07-01')
    and date(l.requested_at) < date(current_date())
'''

In [7]:
df_funil = pandas_gbq.read_gbq(query, project_id=project_id)
df_funil

Downloading: 100%|██████████|


,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,tipo_contrato,product_nm,...,person_deemed_income,pre_analysis_result,lead_elegivel,proposta_iniciada,proposta_enviada,proposta_aprovada,proposta_ativada,is_activeted,first_comunicacao_date,first_competencia_date
0,4624290,2026-09-09,2026-09-09,NaT,NaT,NaT,NaT,NaT,PF,None,...,9110.500000000,APROVAR,True,False,False,False,False,False,NaT,NaT
1,4624229,2026-09-09,2026-09-09,NaT,NaT,NaT,NaT,NaT,PF,None,...,1164.500000000,APROVAR,True,False,False,False,False,False,NaT,NaT
2,4627471,2026-09-09,2026-09-09,NaT,NaT,NaT,NaT,NaT,PF,None,...,1507.000000000,APROVAR,True,False,False,False,False,False,NaT,NaT
3,4623993,2026-09-09,2026-09-09,NaT,NaT,NaT,2026-09-09,NaT,PF,None,...,2260.500000000,REPROVAR,False,False,False,False,False,False,NaT,NaT
4,4628543,2026-09-09,2026-09-09,NaT,NaT,NaT,2026-09-09,NaT,PF,None,...,0E-9,REPROVAR,False,False,False,False,False,False,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608682,4697717,2026-09-23,2026-09-23,2026-09-25,2026-09-25,NaT,NaT,NaT,PF,Flex,...,16371.500000000,APROVAR,True,True,True,True,False,False,NaT,NaT
608683,4699879,2026-09-23,2026-09-23,2026-09-24,2026-09-24,2026-09-24,NaT,NaT,PF,Fit,...,7329.500000000,APROVAR,True,True,True,True,True,True,NaT,NaT
608684,4697793,2026-09-23,2026-09-23,2026-09-23,NaT,NaT,2026-09-23,NaT,PF,Fit,...,35346.000000000,DERIVAR,True,True,False,False,False,False,NaT,NaT
608685,4698732,2026-09-23,2026-09-23,2026-09-23,NaT,NaT,2026-10-24,NaT,PF,Smart,...,1781.000000000,APROVAR,True,True,False,False,False,False,NaT,NaT


In [8]:
df_funil.groupby("bureau_nm_ajust", dropna=False).size()

bureau_nm_ajust
BLEND3                  111476
BLEND3_4                120813
BLEND4                  374868
BLEND_REGRESSAO_2026      1278
NaN                        252
dtype: int64

In [9]:
df_funil.groupby("bureau_nm", dropna=False).size()

bureau_nm
                            77
BLEND3_3                104775
BLEND3_4                116967
BLEND_4                 349779
BLEND_REGRESSAO_2026      1188
BVS_CUSTOM                 482
BVS_CUSTOM_V2             6073
HFT1                      3234
HVA3                     23823
HVA4                      2037
NaN                        252
dtype: int64

In [10]:
df_funil.groupby("bureau_nm_ajust", dropna=False).size()

bureau_nm_ajust
BLEND3                  111476
BLEND3_4                120813
BLEND4                  374868
BLEND_REGRESSAO_2026      1278
NaN                        252
dtype: int64

In [11]:
df_funil.groupby("modeloBlend", dropna=False).size()

modeloBlend
BLEND3_3                110250
BLEND3_4                121832
BLEND_4                 374571
BLEND_REGRESSAO_2026      1485
BVS_CUSTOM_V2              165
HFT1                       132
NaN                        252
dtype: int64

In [12]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

df_funil["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

In [13]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "9.E-BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "2.A",
    "3.B+",
    "3.B+",
    "4.B",
    "4.B",
    "5.C",
    "6.D+",
    "7.D",
    "7.D",
    "8.E",
    "8.E",
]

df_funil["rating_pol_blend4"] = np.select(conditions, choices, default=None)
df_funil["rating_pol_blend4"] = df_funil["rating_pol_blend4"].fillna("N/I")

In [14]:
df_funil.groupby("rating_pol_blend4", dropna=False).size()

rating_pol_blend4
1.A+        20837
2.A         54022
3.B+        59246
4.B         61975
5.C         31666
6.D+        32510
7.D         66270
8.E        163189
9.E-BVS     10504
N/I        108468
dtype: int64

In [15]:
df_funil.to_csv(ANALYTICS_DIR/"df_funil_blend4.csv", index=False)

In [16]:
df_funil[df_funil["contract_id"]==4358668][["contract_id", "modeloBlend", "bureau_nm", "modelo_blend", "bureau_nm_ajust", "is_fallback"]]

,contract_id,modeloBlend,bureau_nm,modelo_blend,bureau_nm_ajust,is_fallback
74894,4358668,BLEND_4,BLEND3_3,BLEND_4,BLEND4,0


In [17]:
df_blend4_dev = pd.read_csv(ANALYTICS_DIR/"df_predict_blend4.csv")

In [18]:
df_blend4_dev.columns.tolist()

['CPF_CNPJ',
 'contract_id',
 'requested_at',
 'data_ultima_consulta',
 'qtd_proponentes',
 'codigo_cidade_ibge',
 'score_imobiliaria',
 'model',
 'id_consulta',
 'id_funcionalidade',
 'documento',
 'rating',
 'result',
 'message_rendaConsiderada',
 'income',
 'message_rendaUtilizada',
 'message_fatorCorrecaoRenda',
 'message_limiteCalculadoContrato',
 'qtde_restricoes__consulta_realizada',
 'message_restricoesPorModalidadeContrato',
 'message_regra_estouro_limite',
 'message_valorRestricaoTotal',
 'message_regua',
 'message_score',
 'message_blendRegressaoPredict',
 'message_decisao',
 'message_classificacao',
 'message_existeErroSerasa',
 'message_existeErroPowerCurve',
 'message_existeErroBvsCustom',
 'message_motor',
 'message_regras_situacao_cadastral_serasa',
 'score_proposto__bvs',
 'message_ratings_BLEND_4',
 'message_feature_flags_analiseAutomaticaDeCredito',
 'message_feature_flags_modeloBlend',
 'flag_tem__contratos_anteriores',
 'flag_teve_boleto_atrasado__contratos_anterio

In [19]:
df_blend4_dev[df_blend4_dev["contract_id"]==4358668][["contract_id", "model", "message_decisao", "message_scores_HVA4", "message_ratings_BVS_CUSTOM", "pessoa_scores_HVA4"]]

,contract_id,model,message_decisao,message_scores_HVA4,message_ratings_BVS_CUSTOM,pessoa_scores_HVA4


In [20]:
df_funil[df_funil["is_fallback"]==1]

,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,tipo_contrato,product_nm,...,lead_elegivel,proposta_iniciada,proposta_enviada,proposta_aprovada,proposta_ativada,is_activeted,first_comunicacao_date,first_competencia_date,rating_cl_pol_blend4,rating_pol_blend4
10,4628233,2026-09-09,2026-09-09,NaT,NaT,NaT,2026-09-09,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
130,4845566,2026-10-23,2026-10-23,2026-10-23,2026-10-23,NaT,NaT,NaT,PF,Smart Plus,...,True,True,True,True,False,False,NaT,NaT,None,N/I
157,4830012,2026-10-21,2026-10-21,NaT,NaT,NaT,2026-10-21,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
189,4834109,2026-10-21,2026-10-21,2026-10-21,2026-10-21,2026-10-22,NaT,NaT,PF,Fit,...,True,True,True,True,True,True,NaT,NaT,None,N/I
212,4840444,2026-10-22,2026-10-22,NaT,NaT,NaT,2026-10-22,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608396,4614506,2026-09-04,2026-09-04,NaT,NaT,NaT,2026-09-04,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
608398,4614051,2026-09-04,2026-09-04,NaT,NaT,NaT,2026-09-04,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
608421,4614177,2026-09-04,2026-09-04,2026-09-04,2026-09-04,2026-09-14,NaT,NaT,PF,Flex,...,True,True,True,True,True,True,NaT,NaT,None,N/I
608516,4860550,2026-10-27,2026-10-27,NaT,NaT,NaT,NaT,NaT,PF,None,...,True,False,False,False,False,False,NaT,NaT,None,N/I


In [21]:
df_funil.groupby("qtd_proponentes", dropna=False).size()

qtd_proponentes
1       454115
2        19616
3         1857
4          243
5           25
6            3
8            1
<NA>    132827
dtype: int64

In [22]:
df_funil.columns

Index(['contract_id', 'dt_lead', 'requested_at', 'iniciada_at', 'enviada_at',
       'activated_at', 'cancelled_at', 'dt_saida', 'tipo_contrato',
       'product_nm', 'modeloBlend', 'bureau_nm', 'modelo_blend',
       'bureau_nm_ajust', 'is_fallback', 'total_rental_value_informed_nr',
       'rental_value_nr', 'agency_id', 'contract_city_nm', 'person_age',
       'qtd_proponentes', 'score_imobiliaria',
       'person_restriction_total_value', 'bvs_cust_score_nr',
       'blend_regressao_predict_nr', 'rating_score_ds', 'person_deemed_income',
       'pre_analysis_result', 'lead_elegivel', 'proposta_iniciada',
       'proposta_enviada', 'proposta_aprovada', 'proposta_ativada',
       'is_activeted', 'first_comunicacao_date', 'first_competencia_date',
       'rating_cl_pol_blend4', 'rating_pol_blend4'],
      dtype='object')

In [23]:
df_funil[df_funil["qtd_proponentes"].isna()].groupby("modeloBlend", dropna=False).size()

modeloBlend
BLEND3_3                36208
BLEND3_4                27635
BLEND_4                 68374
BLEND_REGRESSAO_2026      389
BVS_CUSTOM_V2              57
HFT1                       26
NaN                       138
dtype: int64